<a href="https://colab.research.google.com/github/YenlingPeng/T-Brain_AI_esunbank/blob/Ying/20251025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [19]:
import os
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dir_path = "../preliminary_data/"
df_txn, df_alert, df_test = LoadCSV(dir_path)
df_X = PreProcessing(df_txn)
X_train, X_test, y_train = TrainTestSplit(df_X, df_alert, df_test)
y_pred = Modeling(X_train, y_train, X_test)
out_path = "result.csv"
OutputCSV(out_path, df_test, X_test, y_pred)

In [3]:
dir_path = "/content/drive/MyDrive/2025_alert_account/data"
def LoadCSV(dir_path):
    """
    讀取挑戰賽提供的3個資料集：交易資料、警示帳戶註記、待預測帳戶清單
    Args:
        dir_path (str): 資料夾，請把上述3個檔案放在同一個資料夾

    Returns:
        df_txn: 交易資料 DataFrame
        df_alert: 警示帳戶註記 DataFrame
        df_test: 待預測帳戶清單 DataFrame
    """
    df_txn = pd.read_csv(os.path.join(dir_path, 'acct_transaction.csv'))
    df_alert = pd.read_csv(os.path.join(dir_path, 'acct_alert.csv'))
    df_test = pd.read_csv(os.path.join(dir_path, 'acct_predict.csv'))

    print("(Finish) Load Dataset.")
    return df_txn, df_alert, df_test

In [4]:
df_txn, df_alert, df_test = LoadCSV(dir_path)

(Finish) Load Dataset.


In [ ]:
# 範例的preprocessing

"""
def PreProcessing(df):
    """
    資料處理的範例程式，計算每個帳戶的一些統計量，當作模型因子
    參賽者可自行發想、設計自己的因子
    """
    # 1. 'total_send/recv_amt': total amount sent/received by each acct
    send = df.groupby('from_acct')['txn_amt'].sum().rename('total_send_amt')
    recv = df.groupby('to_acct')['txn_amt'].sum().rename('total_recv_amt')

    # 2. max, min, avg txn_amt for each account
    max_send = df.groupby('from_acct')['txn_amt'].max().rename('max_send_amt')
    min_send = df.groupby('from_acct')['txn_amt'].min().rename('min_send_amt')
    avg_send = df.groupby('from_acct')['txn_amt'].mean().rename('avg_send_amt')

    max_recv = df.groupby('to_acct')['txn_amt'].max().rename('max_recv_amt')
    min_recv = df.groupby('to_acct')['txn_amt'].min().rename('min_recv_amt')
    avg_recv = df.groupby('to_acct')['txn_amt'].mean().rename('avg_recv_amt')

    df_result = pd.concat([max_send, min_send, avg_send, max_recv, min_recv, avg_recv, send, recv], axis=1).fillna(0).reset_index()
    df_result.rename(columns={'index': 'acct'}, inplace=True)

    # 3. 'is_esun': is esun account or not
    df_from = df[['from_acct', 'from_acct_type']].rename(columns={'from_acct': 'acct', 'from_acct_type': 'is_esun'})
    df_to = df[['to_acct', 'to_acct_type']].rename(columns={'to_acct': 'acct', 'to_acct_type': 'is_esun'})
    df_acc = pd.concat([df_from, df_to], ignore_index=True).drop_duplicates().reset_index(drop=True)

    # 4. merge (1), (2), and (3)
    df_result = pd.merge(df_result, df_acc, on='acct', how='left')
    print("(Finish) PreProcessing.")
    return df_result
"""

In [21]:
def calculate_all_acct_features(df_txn: pd.DataFrame) -> pd.DataFrame:
    """
    計算所有帳戶的五個指定特徵 (以 from_acct 為主體)，並將結果與帳戶類型合併。

    Args:
        df_txn: 包含交易資料的 DataFrame，必須包含以下欄位：
                'from_acct', 'from_acct_type', 'to_acct', 'to_acct_type',
                'txn_amt', 'txn_date', 'txn_time'

    Returns:
        一個新的 DataFrame，包含每個 from_acct 的五個特徵。
    """

    # ----------------------------------------------------------------------
    # 步驟 1: 基礎統計量 (總筆數、活躍天數、總出款金額)
    # ----------------------------------------------------------------------

    # 總出款筆數 (以 from_acct 為主體)
    out_tx_count = df_txn.groupby('from_acct')['from_acct'].count().rename('total_tx_count')
    # 活躍天數
    active_days = df_txn.groupby('from_acct')['txn_date'].nunique().rename('active_days')
    # 總出款金額
    out_amount = df_txn.groupby('from_acct')['txn_amt'].sum().rename('out_amount')

    # 合併基礎統計
    base_stats = pd.concat([out_tx_count, active_days, out_amount], axis=1).fillna(0)

    # ----------------------------------------------------------------------
    # 步驟 2: 計算每日平均出款筆數 (avg_out_tx_per_day)
    # ----------------------------------------------------------------------
    base_stats['avg_out_tx_per_day'] = base_stats['total_tx_count'] / base_stats['active_days']

    # ----------------------------------------------------------------------
    # 步驟 3: 計算出入金比例 (out_in_amount_ratio)
    # ----------------------------------------------------------------------

    # 總入款金額 (以 to_acct 為主體)
    in_amount = df_txn.groupby('to_acct')['txn_amt'].sum().rename('in_amount')
    in_amount.index.name = 'from_acct' # 為了合併，將索引名稱改為 from_acct

    # 將出款金額和入款金額合併到所有 from_acct 的基礎統計上
    amount_features = pd.concat([base_stats['out_amount'], in_amount], axis=1).fillna(0)

    # 計算 out_in_amount_ratio
    base_stats['out_in_amount_ratio'] = amount_features['out_amount'] / (amount_features['in_amount'] + 1)

    # ----------------------------------------------------------------------
    # 步驟 4: 計算同日重複交易比例 (same_day_repeat_tx_ratio)
    # ----------------------------------------------------------------------

    # 計算每個 (from_acct, to_acct, txn_date) 群組的交易筆數
    df_txn['tx_group_count'] = df_txn.groupby(['from_acct', 'to_acct', 'txn_date'])['from_acct'].transform('count')

    # 標記重複交易：tx_group_count > 1
    df_txn['is_repeat_tx'] = np.where(df_txn['tx_group_count'] > 1, 1, 0)

    # 計算每個 from_acct 的重複交易筆數總和
    repeat_tx_count = df_txn.groupby('from_acct')['is_repeat_tx'].sum()

    # 計算比例
    base_stats['same_day_repeat_tx_ratio'] = repeat_tx_count / base_stats['total_tx_count']

    # ----------------------------------------------------------------------
    # 步驟 5: 計算跨行交易比例 (cross_bank_ratio)
    # ----------------------------------------------------------------------

    # 跨行交易定義：from_acct_type != to_acct_type
    df_txn['is_cross_bank'] = np.where(df_txn['from_acct_type'] != df_txn['to_acct_type'], 1, 0)

    # 計算每個 from_acct 的跨行交易筆數
    cross_bank_count = df_txn.groupby('from_acct')['is_cross_bank'].sum()

    # 計算比例
    base_stats['cross_bank_ratio'] = cross_bank_count / base_stats['total_tx_count']

    # ----------------------------------------------------------------------
    # 步驟 6: 計算夜間交易比例 (night_tx_ratio)
    # ----------------------------------------------------------------------

    # 夜間交易定義：交易時間在 23:00–05:00 的筆數比例
    # txn_time 是數值型，格式為 HHMMSS，所以 230000 <= txn_time 或 txn_time <= 050000
    df_txn['is_night_tx'] = np.where(
        (df_txn['txn_time'] > 230000) | (df_txn['txn_time'] < 50000),
        1,
        0
    )

    # 計算每個 from_acct 的夜間交易筆數
    night_tx_count = df_txn.groupby('from_acct')['is_night_tx'].sum()

    # 計算比例
    base_stats['night_tx_ratio'] = night_tx_count / base_stats['total_tx_count']

    # ----------------------------------------------------------------------
    # 步驟 7: 合併帳戶類型資訊 (is_esun)
    # ----------------------------------------------------------------------

    # 取得 from_acct 的帳戶類型 (假設一個 from_acct 只有一種 from_acct_type)
    acct_type_map = df_txn.drop_duplicates(subset=['from_acct'])[['from_acct', 'from_acct_type']]
    acct_type_map.set_index('from_acct', inplace=True)

    # 將帳戶類型合併到結果 DataFrame
    df_result = base_stats.join(acct_type_map).reset_index()
    df_result.rename(columns={'index': 'from_acct', 'from_acct_type': 'is_esun'}, inplace=True)

    # ----------------------------------------------------------------------
    # 步驟 8: 篩選玉山匯款帳戶 (is_esun == '01')
    # ----------------------------------------------------------------------

    final_df = df_result[df_result['is_esun'] == '01'].copy()

    # 整理最終 DataFrame，只保留需要的欄位
    final_df = final_df[[
        'from_acct',
        'avg_out_tx_per_day',
        'out_in_amount_ratio',
        'same_day_repeat_tx_ratio',
        'cross_bank_ratio',
        'night_tx_ratio'
    ]]

    return final_df

In [22]:
df_X = calculate_all_acct_features(df_txn)

TypeError: '>' not supported between instances of 'str' and 'int'

In [12]:
def TrainTestSplit(df, df_alert, df_test):
    """
    切分訓練集及測試集，並為訓練集的帳戶標上警示label (0為非警示、1為警示)

    備註:
        1. 測試集為待預測帳戶清單，你需要預測它們
        2. 此切分僅為範例，較標準的做法是基於訓練集再且分成train和validation，請有興趣的參賽者自行切分
        3. 由於待預測帳戶清單僅為玉山戶，所以我們在此範例僅使用玉山帳戶做訓練
    """
    X_train = df[(~df['acct'].isin(df_test['acct'])) & (df['is_esun']==1)].drop(columns=['is_esun']).copy()
    y_train = X_train['acct'].isin(df_alert['acct']).astype(int)
    X_test = df[df['acct'].isin(df_test['acct'])].drop(columns=['is_esun']).copy()

    print(f"(Finish) Train-Test-Split")
    return X_train, X_test, y_train

In [13]:
X_train, X_test, y_train = TrainTestSplit(df_X, df_alert, df_test)

KeyError: 'acct'

In [ ]:
def Modeling(X_train, y_train, X_test):
    """
    Decision Tree的範例程式，參賽者可以在這裡實作自己需要的方法
    """
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train.drop(columns=['acct']), y_train)
    y_pred = model.predict(X_test.drop(columns=['acct']))

    print(f"(Finish) Modeling")
    return y_pred

In [ ]:
def OutputCSV(path, df_test, X_test, y_pred):
    """
    根據測試資料集及預測結果，產出預測結果之CSV，該CSV可直接上傳於TBrain
    """
    df_pred = pd.DataFrame({
        'acct': X_test['acct'].values,
        'label': y_pred
    })

    df_out = df_test[['acct']].merge(df_pred, on='acct', how='left')
    df_out.to_csv(path, index=False)

    print(f"(Finish) Output saved to {path}")